# Heating Oil / ULSD Flat Price Contracts — Open Interest EDA

Analyze the relative size of each Heating Oil flat price contract code from `disaggregated_combined.csv`.

Flat price codes (from `docs/heating_oil_cot_mapping.md`):
- `022651` — NY HARBOR ULSD - NEW YORK MERCANTILE EXCHANGE (benchmark)
- `022653` — HEATING OIL AVG PRICE OPTIONS - NEW YORK MERCANTILE EXCHANGE
- `022654` — EUR STYLE HEATING OIL OPTIONS - NEW YORK MERCANTILE EXCHANGE
- `022A05` — GULF COAST ULSD SWAP - NEW YORK MERCANTILE EXCHANGE
- `02265V` — GASOIL (ICE) SWAP - NEW YORK MERCANTILE EXCHANGE
- `022A24` — GASOIL (ICE) MINI CAL SWAP - NEW YORK MERCANTILE EXCHANGE
- `022A46` — GASOIL AVG PRICE OPTIONS - NEW YORK MERCANTILE EXCHANGE
- `02265J` — SING GASOIL SWAP - NEW YORK MERCANTILE EXCHANGE
- `022A22` — SING GASOIL BALMO SWAP - NEW YORK MERCANTILE EXCHANGE

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
df = pd.read_csv("../../downloads/cftc/disaggregated_combined.csv", low_memory=False)

HO_FLAT_PRICE_CODES = [
    "022651",  # NY HARBOR ULSD - NYMEX (benchmark)
    "022653",  # HEATING OIL AVG PRICE OPTIONS - NYMEX
    "022654",  # EUR STYLE HEATING OIL OPTIONS - NYMEX
    "022A05",  # GULF COAST ULSD SWAP - NYMEX
    "02265V",  # GASOIL (ICE) SWAP - NYMEX (added per Marouen)
    "022A24",  # GASOIL (ICE) MINI CAL SWAP - NYMEX (added per Marouen)
    "022A46",  # GASOIL AVG PRICE OPTIONS - NYMEX (added per Marouen)
    "02265J",  # SING GASOIL SWAP - NYMEX (added per Marouen)
    "022A22",  # SING GASOIL BALMO SWAP - NYMEX (added per Marouen)
]

ho = df[df["cftc_contract_market_code"].isin(HO_FLAT_PRICE_CODES)].copy()
ho["report_date_as_yyyy_mm_dd"] = pd.to_datetime(ho["report_date_as_yyyy_mm_dd"])
ho["open_interest_all"] = pd.to_numeric(ho["open_interest_all"], errors="coerce")
print(f"HO flat price rows: {len(ho):,}")
print(f"Date range: {ho['report_date_as_yyyy_mm_dd'].min().date()} to {ho['report_date_as_yyyy_mm_dd'].max().date()}")
print(f"Codes found: {sorted(ho['cftc_contract_market_code'].unique())}")

HO flat price rows: 1,630
Date range: 2006-06-13 to 2026-03-31
Codes found: ['022651', '022653', '022654', '02265J', '02265V', '022A05', '022A22', '022A24', '022A46']


## Average Open Interest by Contract Code

In [3]:
code_labels = (
    ho.groupby("cftc_contract_market_code")["market_and_exchange_names"]
    .first()
    .to_dict()
)

avg_oi = (
    ho.groupby("cftc_contract_market_code")["open_interest_all"]
    .mean()
    .sort_values(ascending=False)
    .to_frame("avg_open_interest")
)
avg_oi["contract_name"] = avg_oi.index.map(code_labels)
avg_oi["source"] = "CFTC"
avg_oi["pct_of_total"] = (avg_oi["avg_open_interest"] / avg_oi["avg_open_interest"].sum() * 100)
avg_oi["avg_open_interest"] = avg_oi["avg_open_interest"].round(0).astype(int)
avg_oi["pct_of_total"] = avg_oi["pct_of_total"].round(2)

avg_oi[["contract_name", "source", "avg_open_interest", "pct_of_total"]]

,contract_name,source,avg_open_interest,pct_of_total
cftc_contract_market_code,,,,
022651,"NO. 2 HEATING OIL, N.Y. HARBOR - NEW YORK MERC...",CFTC,358528,70.27
022653,HEATING OIL AVG PRICE OPTIONS - NEW YORK MERCA...,CFTC,93322,18.29
02265J,SING GASOIL SWAP - NEW YORK MERCANTILE EXCHANGE,CFTC,15813,3.10
022654,EUR STYLE HEATING OIL OPTIONS - NEW YORK MERCA...,CFTC,13931,2.73
022A05,GULF COAST ULSD SWAP - NEW YORK MERCANTILE EXC...,CFTC,13774,2.70
022A24,GASOIL (ICE) MINI CAL SWAP - NEW YORK MERCANTI...,CFTC,4914,0.96
02265V,GASOIL (ICE) SWAP - NEW YORK MERCANTILE EXCHANGE,CFTC,4375,0.86
022A46,GASOIL AVG PRICE OPTIONS - NEW YORK MERCANTILE...,CFTC,3490,0.68
022A22,SING GASOIL BALMO SWAP - NEW YORK MERCANTILE E...,CFTC,2033,0.40


## Last Report Date by Contract Code

In [4]:
last_report = (
    ho.groupby("cftc_contract_market_code")
    .agg(
        contract_name=("market_and_exchange_names", "first"),
        last_report_date=("report_date_as_yyyy_mm_dd", "max"),
        last_report_oi=("open_interest_all", "last"),
    )
    .sort_values("last_report_date", ascending=False)
)
last_report["source"] = "CFTC"
last_report["last_report_date"] = last_report["last_report_date"].dt.date
last_report

,contract_name,last_report_date,last_report_oi,source
cftc_contract_market_code,,,,
022651,"NO. 2 HEATING OIL, N.Y. HARBOR - NEW YORK MERC...",2026-03-31,250551,CFTC
02265J,SING GASOIL SWAP - NEW YORK MERCANTILE EXCHANGE,2017-08-22,17364,CFTC
022A05,GULF COAST ULSD SWAP - NEW YORK MERCANTILE EXC...,2012-10-02,18841,CFTC
02265V,GASOIL (ICE) SWAP - NEW YORK MERCANTILE EXCHANGE,2012-09-04,3352,CFTC
022A22,SING GASOIL BALMO SWAP - NEW YORK MERCANTILE E...,2012-07-31,1526,CFTC
022653,HEATING OIL AVG PRICE OPTIONS - NEW YORK MERCA...,2012-04-24,48440,CFTC
022A24,GASOIL (ICE) MINI CAL SWAP - NEW YORK MERCANTI...,2011-09-27,3850,CFTC
022A46,GASOIL AVG PRICE OPTIONS - NEW YORK MERCANTILE...,2010-12-28,3539,CFTC
022654,EUR STYLE HEATING OIL OPTIONS - NEW YORK MERCA...,2008-12-22,10085,CFTC


## Open Interest Share — Pie Chart

In [5]:
fig = px.pie(
    avg_oi.reset_index(),
    values="avg_open_interest",
    names="cftc_contract_market_code",
    hover_data=["contract_name"],
    title="Heating Oil Flat Price Contracts — Average Open Interest Share",
)
fig.update_traces(textinfo="label+percent", textposition="outside")
fig.show()

## Open Interest Over Time by Contract Code

In [6]:
fig = px.line(
    ho.sort_values("report_date_as_yyyy_mm_dd"),
    x="report_date_as_yyyy_mm_dd",
    y="open_interest_all",
    color="cftc_contract_market_code",
    hover_data=["market_and_exchange_names"],
    title="Heating Oil Flat Price Contracts — Open Interest Over Time",
    labels={
        "report_date_as_yyyy_mm_dd": "Report Date",
        "open_interest_all": "Open Interest",
        "cftc_contract_market_code": "Contract Code",
    },
)
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.3))
fig.show()

## Open Interest Over Time — Stacked Area (Percentage)

In [7]:
pivot = ho.pivot_table(
    index="report_date_as_yyyy_mm_dd",
    columns="cftc_contract_market_code",
    values="open_interest_all",
    aggfunc="sum",
).fillna(0)

pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig = go.Figure()
for col in pct.columns:
    label = code_labels.get(col, col)
    fig.add_trace(go.Scatter(
        x=pct.index, y=pct[col],
        name=col,
        hovertext=label,
        stackgroup="one",
        groupnorm="percent",
    ))
fig.update_layout(
    title="Heating Oil Flat Price Contracts — OI Share Over Time (%)",
    xaxis_title="Report Date",
    yaxis_title="% of Total HO Flat Price OI",
    legend=dict(orientation="h", yanchor="bottom", y=-0.3),
)
fig.show()

## Summary Table — Latest Reporting Date

In [8]:
latest_date = ho["report_date_as_yyyy_mm_dd"].max()
latest = ho[ho["report_date_as_yyyy_mm_dd"] == latest_date].copy()

summary = (
    latest.groupby("cftc_contract_market_code")
    .agg(
        contract_name=("market_and_exchange_names", "first"),
        open_interest=("open_interest_all", "sum"),
    )
    .sort_values("open_interest", ascending=False)
)
summary["pct_of_total"] = (summary["open_interest"] / summary["open_interest"].sum() * 100).round(2)

print(f"Latest report date: {latest_date.date()}")
summary

Latest report date: 2026-03-31


,contract_name,open_interest,pct_of_total
cftc_contract_market_code,,,
022651,NY HARBOR ULSD - NEW YORK MERCANTILE EXCHANGE,250551,100.0
